In [1]:
import sys
!{sys.executable} -m pip install pandas dash plotly

In [2]:
import pandas as pd
import dash
import plotly.express as px
print("Packages installed successfully")

Packages installed successfully


In [3]:
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"

spacex_df = pd.read_csv(dataset_url)
spacex_df.to_csv("spacex_launch_dash.csv", index=False)

spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [4]:
import urllib.request

app_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/t4-Vy4iOU19i8y6E3Px_ww/spacex-dash-app.py"
urllib.request.urlretrieve(app_url, "spacex-dash-app.py")
print("File downloaded successfully")

File downloaded successfully


In [22]:
%%writefile spacex-dash-app.py

# Import required libraries
import pandas as pd
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the SpaceX data into pandas dataframe
spacex_df = pd.read_csv("spacex_launch_dash.csv")

max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create dropdown options
site_options = [{'label': 'All Sites', 'value': 'ALL'}]

for site in sorted(spacex_df['Launch Site'].unique()):
    site_options.append({'label': site, 'value': site})

# Create a dash application
app = dash.Dash(__name__)

# Create an app layout
app.layout = html.Div(children=[

    html.H1(
        'SpaceX Launch Records Dashboard',
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'font-size': 40
        }
    ),

    # TASK 1: Dropdown list to enable Launch Site selection
    dcc.Dropdown(
        id='site-dropdown',
        options=site_options,
        value='ALL',
        placeholder='Select a Launch Site here',
        searchable=True,
        style={
            'width': '80%',
            'margin': 'auto'
        }
    ),

    html.Br(),

    # TASK 2: Pie chart
    html.Div(dcc.Graph(id='success-pie-chart')),

    html.Br(),

    html.P("Payload range (Kg):"),

    # TASK 3: Range Slider to select payload range
    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={
            0: '0',
            2500: '2500',
            5000: '5000',
            7500: '7500',
            10000: '10000'
        },
        value=[min_payload, max_payload]
    ),

    html.Br(),

    # TASK 4: Scatter chart
    html.Div(dcc.Graph(id='success-payload-scatter-chart'))

])


# TASK 2: Callback function for pie chart
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):

    if entered_site == 'ALL':
        fig = px.pie(
            spacex_df,
            values='class',
            names='Launch Site',
            title='Total Success Launches by Site'
        )
        return fig

    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]

        fig = px.pie(
            filtered_df,
            names='class',
            title='Total Success and Failed Launches for {}'.format(entered_site)
        )
        return fig


# TASK 4: Callback function for scatter chart
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [
        Input(component_id='site-dropdown', component_property='value'),
        Input(component_id='payload-slider', component_property='value')
    ]
)
def get_scatter_chart(entered_site, payload_range):

    low_payload = payload_range[0]
    high_payload = payload_range[1]

    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= low_payload) &
        (spacex_df['Payload Mass (kg)'] <= high_payload)
    ]

    if entered_site == 'ALL':
        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title='Correlation between Payload and Success for All Sites'
        )
        return fig

    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]

        fig = px.scatter(
            site_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title='Correlation between Payload and Success for {}'.format(entered_site)
        )
        return fig


# Run the app
if __name__ == '__main__':
    app.run(debug=False, port=8060, use_reloader=False)

Overwriting spacex-dash-app.py


In [23]:
import subprocess
import re

PORT = "8060"

result = subprocess.run(
    f'netstat -ano | findstr :{PORT}',
    shell=True,
    capture_output=True,
    text=True
)

pids = set()

for line in result.stdout.splitlines():
    parts = line.split()
    if len(parts) >= 5:
        pid = parts[-1]
        if pid.isdigit():
            pids.add(pid)

for pid in pids:
    subprocess.run(
        f'taskkill /PID {pid} /F',
        shell=True,
        capture_output=True,
        text=True
    )
    print(f"Process {pid} killed")

if not pids:
    print(f"No active process found on port {PORT}")

Process 9340 killed


In [24]:
import sys
!{sys.executable} -u spacex-dash-app.py

^C
